# Deteksi gelombang P, Q, R, S, T pada ECG dengan DWT (db4, from scratch)

**Dataset:** MIT-BIH Arrhythmia Database (PhysioNet), record 122 (`.dat`, `.hea`, `.atr`).
**DWT:** koefisien LPF/HPF db4 diketik manual, dekomposisi Mallat (a trous) ditulis sendiri tanpa `pywt`/`scipy`.
`wfdb` hanya dipakai untuk **membaca file** PhysioNet.

**Cara pakai:** jalankan sel dari atas ke bawah. Pada sel unggah, pilih ketiga file sekaligus.
Panel kiri: `NData` = jumlah sampel, `Start` = sampel awal, `Action` = jalankan. `T1..T8` dan `Delay1..Delay8` = delay filter Q_j
(otomatis; hilangkan centang untuk mengisi manual).

**Rujukan rumus:** Mallat (1989); Martinez et al. (2004), IEEE TBME 51(4); Hikmah, Arifin & Sardjono (2020), JUTI 18(2).

In [25]:
# %pip install -q wfdb ipywidgets
import numpy as np
import matplotlib.pyplot as plt
# %matplotlib inline


In [26]:
import os
REC = os.path.join(os.getcwd(), '122')
print('Record:', REC, '| ada .dat:', os.path.exists(REC + '.dat'))

Record: c:\Users\HP\OneDrive\Desktop\Semester 5\Komunikasi Data\Pertemuan 6\122 | ada .dat: True


In [27]:
# ============================================================
# 1. DWT FROM SCRATCH (tanpa pywt / scipy.signal) - wavelet db4
# ============================================================
import numpy as np

# --- Koefisien LPF db4 h[n] (diketik manual, skala ortonormal: sum = sqrt(2))
h_db4 = np.array([ 0.2303778133088964,  0.7148465705529154,  0.6308807679298587,
                  -0.0279837694168599, -0.1870348117190931,  0.0308413818355607,
                   0.0328830116668852, -0.0105974017850690])
L = len(h_db4)
# --- HPF dibangkitkan sendiri: g[n] = (-1)^n * h[L-1-n]
g_db4 = np.array([(-1)**n * h_db4[L-1-n] for n in range(L)])

# Normalisasi gain DC = 1 per level (dibagi sqrt(2)) agar amplitudo tiap skala sebanding
H = h_db4 / np.sqrt(2)
G = g_db4 / np.sqrt(2)
JMAX = 8          # jumlah skala Q1..Q8

def konv(x, f):
    """Konvolusi penuh y[k] = sum_m f[m] x[k-m] (ditulis manual, tanpa np.convolve)."""
    y = np.zeros(len(x) + len(f) - 1)
    for m, c in enumerate(f):
        if c != 0.0:
            y[m:m + len(x)] += c * x
    return y

def upsample(f, m):
    """Sisipkan (m-1) nol di antara koefisien -> filter algoritma a trous."""
    out = np.zeros((len(f) - 1) * m + 1)
    out[::m] = f
    return out

def build_Q(J=JMAX):
    """Filter bank ekuivalen Mallat:
       Q_j = G(2^(j-1) w) * H(2^(j-2) w) * ... * H(w)   (tanpa decimation)."""
    Qs, a = [], np.array([1.0])
    for j in range(1, J + 1):
        Qs.append(konv(a, upsample(G, 2 ** (j - 1))))
        a = konv(a, upsample(H, 2 ** (j - 1)))
    return Qs

Q = build_Q()

def delay_of(q):
    """Delay filter (pusat energi impuls respons) -> nilai T_j."""
    n = np.arange(len(q))
    return int(round(np.sum(n * q**2) / np.sum(q**2)))

T_auto = [delay_of(q) for q in Q]
D_auto = [max(T_auto) - t for t in T_auto]          # Delay_j = Tmax - T_j

# tanda (+/-) agar tepi naik sinyal menghasilkan koefisien positif
SGN = []
for q in Q:
    c = np.cumsum(q)
    SGN.append(1.0 if c[np.argmax(np.abs(c))] > 0 else -1.0)

# delay filter lowpass kumulatif A_j = H(w)H(2w)...H(2^(j-1)w) (pusat massa = delay grup DC)
def _lp_delay(J):
    a, out = np.array([1.0]), []
    for j in range(1, J + 1):
        a = konv(a, upsample(H, 2 ** (j - 1)))
        out.append(int(round(np.sum(np.arange(len(a)) * a) / np.sum(a))))
    return out
DA = _lp_delay(JMAX)

def mallat(x, T, D, return_A=False):
    """Koefisien wavelet W_j (j=1..8) yang sudah disejajarkan dengan sinyal asli.
       Alur: a_0 = x ; w_j = a_{j-1} * G_up ; a_j = a_{j-1} * H_up.
       Tiap w_j ditunda D_j, lalu semua digeser Tmax (T_j + D_j = Tmax)."""
    pad = max(len(q) for q in Q) // 2 + 50
    xp = np.pad(x, (pad, pad), mode='reflect')
    Tmax, n = max(T), len(x)
    a, Wj, Aj = xp.copy(), [], []
    for j in range(1, len(Q) + 1):
        y = konv(a, upsample(G, 2 ** (j - 1)))
        a = konv(a, upsample(H, 2 ** (j - 1)))
        idx = np.clip(np.arange(n) + pad + Tmax - D[j - 1], 0, len(y) - 1)
        Wj.append(SGN[j - 1] * y[idx])
        Aj.append(a[np.clip(np.arange(n) + pad + DA[j - 1], 0, len(a) - 1)])
    return (Wj, Aj) if return_A else Wj

# --- pemeriksaan koefisien
print("sum h  =", round(h_db4.sum(), 6), "(harus sqrt2 =", round(np.sqrt(2), 6), ")")
print("sum h^2=", round((h_db4**2).sum(), 6), "(harus 1)   sum g =", round(g_db4.sum(), 6), "(harus 0)")
print("T_j   :", T_auto)
print("Delay :", D_auto)


sum h  = 1.414214 (harus sqrt2 = 1.414214 )
sum h^2= 1.0 (harus 1)   sum g = 0.0 (harus 0)
T_j   : [6, 13, 27, 55, 111, 223, 447, 895]
Delay : [889, 882, 868, 840, 784, 672, 448, 0]


In [28]:
# ============================================================
# 2. BACA DATA + DETEKSI R, Q, S, P, T (zero-crossing & modulus maxima)
# ============================================================
import wfdb
BEAT = set("NLRBAaJSVrFejnE/fQ?")
_cache = {}

def load_record(rec, start, ndata, ch=0):
    key = (rec, start, ndata, ch)
    if key not in _cache:
        r = wfdb.rdrecord(rec, sampfrom=start, sampto=start + ndata)
        a = wfdb.rdann(rec, 'atr', sampfrom=start, sampto=start + ndata)
        ref = np.array([s - start for s, sym in zip(a.sample, a.symbol) if sym in BEAT], int)
        _cache[key] = (r.p_signal[:, ch].astype(float), r.fs, ref, r.sig_name[ch])
    return _cache[key]

def local_ext(w, thr):
    i = np.arange(1, len(w) - 1)
    mx = i[(w[i] > w[i-1]) & (w[i] >= w[i+1]) & (w[i] > thr)]
    mn = i[(w[i] < w[i-1]) & (w[i] <= w[i+1]) & (w[i] < -thr)]
    return mx, mn

def zc_pairs(w, thr, win):
    """Pasangan modulus maxima-minima berlawanan tanda (|w|>thr); zero-crossing di antaranya.
       sign=+1 : max lalu min  -> puncak ke atas ; sign=-1 : min lalu max -> lembah."""
    mx, mn = local_ext(w, thr)
    ext = sorted([(int(i), 1) for i in mx] + [(int(i), -1) for i in mn])
    out = []
    for (i1, s1), (i2, s2) in zip(ext, ext[1:]):
        if s1 != s2 and i2 - i1 <= win:
            k = np.where(w[i1:i2 + 1] * s1 <= 0)[0]
            z = i1 + (int(k[0]) if len(k) else (i2 - i1) // 2)
            if z > i1 and abs(w[z-1]) < abs(w[z]):
                z -= 1
            out.append(dict(z=z, e1=i1, e2=i2, sign=s1, amp=abs(w[i1]) + abs(w[i2])))
    return out

def best_peak(w, a, b, thr):
    """Zero-crossing dengan pasangan modulus maxima-minima terbesar di jendela [a,b]."""
    a, b = max(a, 1), min(b, len(w) - 2)
    if b - a < 4:
        return None
    s = np.sign(w[a:b + 1])
    idx = np.where(s[:-1] * s[1:] < 0)[0] + a
    if len(idx) == 0:
        return None
    edges = [a] + list(idx) + [b]
    best = None
    for t, z in enumerate(idx):
        l, r = edges[t], edges[t + 2]
        li = l + int(np.argmax(np.abs(w[l:z + 1])))
        ri = z + 1 + int(np.argmax(np.abs(w[z + 1:r + 1])))
        al, ar = abs(w[li]), abs(w[ri])
        if min(al, ar) >= thr and (best is None or al + ar > best['score']):
            zz = int(z) if abs(w[z]) <= abs(w[z + 1]) else int(z) + 1
            best = dict(z=zz, e1=li, e2=ri, sign=1 if w[li] > 0 else -1, score=al + ar)
    return best

def refine(A, b):
    """Puncak akhir = ekstremum sinyal aproksimasi A_j di antara modulus maxima-minima."""
    seg = A[b['e1']:b['e2'] + 1]
    k = np.argmax(seg) if b['sign'] == 1 else np.argmin(seg)
    return b['e1'] + int(k)

def detect(x, fs, Wj, P, T, D):
    ms = lambda v: int(round(v * fs / 1000))
    n = len(x)
    w3, w4 = Wj[2], Wj[3]
    res = dict(R=[], Q=[], S=[], P=[], T=[], boxR=[], boxP=[], boxT=[],
               zc=dict(Q=[], S=[], R=[], P=[], T=[]), pol=1,
               thr4=0, thr5=0, thr6=0, pairsR=[], w5=Wj[4], w6=Wj[5])
    # ---------- R : zero-crossing skala 4 (AND skala 3)
    thr4 = P['ThR'] * np.max(np.abs(w4)); res['thr4'] = thr4
    pr = zc_pairs(w4, thr4, ms(100))
    if not pr:
        return res
    npos = sum(p['sign'] == 1 for p in pr)
    pol = 1 if npos >= len(pr) - npos else -1
    res['pol'] = pol
    pr = [p for p in pr if p['sign'] == pol]
    pr3 = zc_pairs(w3, P['ThR'] * 0.5 * np.max(np.abs(w3)), ms(100))
    z3 = np.array([p['z'] for p in pr3 if p['sign'] == pol])
    pr = [p for p in pr if len(z3) and np.min(np.abs(z3 - p['z'])) <= ms(25)]
    pr.sort(key=lambda p: p['z'])
    sel = []
    for p in pr:                                   # periode refrakter
        if sel and p['z'] - sel[-1]['z'] < ms(P['Refr']):
            if p['amp'] > sel[-1]['amp']:
                sel[-1] = p
        else:
            sel.append(p)
    R = []
    for p in sel:
        a, b = max(p['z'] - ms(30), 0), min(p['z'] + ms(30), n)
        R.append(a + int(np.argmax(pol * x[a:b])))
    res['R'] = R; res['pairsR'] = sel
    res['zc']['R'] = [p['z'] for p in sel]
    res['boxR'] = [(p['e1'], p['e2']) for p in sel]
    # ---------- Q dan S : zero-crossing skala 3
    wq = pol * w3
    Qs, Ss = [], []
    for r in R:
        sa = max(r - ms(60), 0)
        pm = sa + int(np.argmax(wq[sa:r + 1]))
        q = None
        for k in range(pm, max(r - ms(80), 1), -1):
            if wq[k-1] < 0 <= wq[k]:
                q = k; break
        if q is None or q >= r:
            q = sa + int(np.argmin(pol * x[sa:r + 1]))
        res['zc']['Q'].append(q)
        q = max(q - 2, 0) + int(np.argmin(pol * x[max(q - 2, 0):min(q + 3, n)]))
        sb = min(r + ms(60), n - 1)
        nm = r + int(np.argmin(wq[r:sb + 1]))
        s = None
        for k in range(nm + 1, min(r + ms(100), n - 1)):
            if wq[k-1] < 0 <= wq[k]:
                s = k; break
        if s is None or s <= r:
            s = r + int(np.argmin(pol * x[r:sb + 1]))
        res['zc']['S'].append(s)
        s = max(s - 2, 0) + int(np.argmin(pol * x[max(s - 2, 0):min(s + 3, n)]))
        Qs.append(q); Ss.append(s)
    res['Q'], res['S'] = Qs, Ss
    # ---------- QRS dipotong (interpolasi linear) agar tidak bocor ke skala 5-6, lalu DWT ulang
    xc = x.copy()
    for q, s in zip(Qs, Ss):
        a, b = max(q - ms(10), 0), min(s + ms(20), n - 1)
        xc[a:b + 1] = np.linspace(xc[a], xc[b], b - a + 1)
    Wc, Ac = mallat(xc, T, D, return_A=True)
    w5, w6 = Wc[4], Wc[5]
    res['w5'], res['w6'] = w5, w6
    # ---------- ambang P/T dari bagian non-QRS
    mask = np.ones(n, bool)
    for r in R:
        mask[max(r - ms(100), 0):r + ms(100)] = False
    ref = lambda w: np.abs(w[mask]).max() if mask.any() else np.abs(w).max()
    res['thr5'], res['thr6'] = P['ThP'] * ref(w5), P['ThT'] * ref(w6)
    # ---------- T (skala 6) lalu P (skala 5)
    Ts = [None] * len(R)
    for i, r in enumerate(R):
        ta = Ss[i] + ms(30)
        tb = r + ms(P['Twin'])
        if i + 1 < len(R):
            tb = min(tb, (r + R[i + 1]) // 2)
        b = best_peak(w6, ta, tb, res['thr6'])
        if b:
            res['zc']['T'].append(b['z']); res['boxT'].append((b['e1'], b['e2']))
            Ts[i] = refine(Ac[4], b)
    for i, r in enumerate(R):
        pb = Qs[i] - ms(15)
        pa = r - ms(P['Pwin'])
        if i > 0:
            pa = max(pa, (Ts[i-1] + ms(40)) if Ts[i-1] is not None else (R[i-1] + r) // 2)
        b = best_peak(w5, pa, pb, res['thr5'])
        if b:
            res['P'].append(refine(Ac[3], b)); res['zc']['P'].append(b['z']); res['boxP'].append((b['e1'], b['e2']))
    res['T'] = [t for t in Ts if t is not None]
    return res

def evaluate(R, ref, fs, tol=0.08):
    R, ref = np.array(R), np.array(ref)
    if len(ref) == 0 or len(R) == 0:
        return None
    t = int(tol * fs)
    se = np.mean([np.any(np.abs(R - b) <= t) for b in ref])
    ppv = np.mean([np.any(np.abs(ref - r) <= t) for r in R])
    return se, ppv

def analyze(x, fs, ref, P, T, D):
    Wj = mallat(x, T, D)
    res = detect(x, fs, Wj, P, T, D)
    res.update(x=x, fs=fs, Wj=Wj, ref=ref, off=P['Start'], metric=evaluate(res['R'], ref, fs))
    return res


In [29]:
# ============================================================
# 3. VISUALISASI (tiap fungsi = satu tab)
# ============================================================
import matplotlib.pyplot as plt

COL = dict(P='tab:green', Q='tab:blue', R='tab:red', S='tab:purple', T='tab:brown')

def _n(res):
    return np.arange(len(res['x'])) + res['off']

def _freq(f, N=4096):
    return np.abs(np.fft.rfft(f, N))

def plot_input(res):
    fig, ax = plt.subplots(figsize=(11, 3.4))
    ax.plot(_n(res), res['x'], lw=.9)
    if len(res['ref']):
        ax.plot(res['ref'] + res['off'], res['x'][res['ref']], 'gx', label='anotasi .atr')
        ax.legend(loc='upper right')
    ax.set(title=f"Input - {res['name']} (fs={res['fs']} Hz, N={len(res['x'])})", xlabel='sampel (n)', ylabel='mV')
    ax.grid(alpha=.3); plt.show()

def plot_mallat(res):
    Wj, x, n = res['Wj'], res['x'], _n(res)
    fig, axs = plt.subplots(len(Wj) + 1, 1, figsize=(11, 12), sharex=True)
    axs[0].plot(n, x, lw=.8); axs[0].set_ylabel('x', rotation=0, labelpad=14)
    axs[0].set_title('Dekomposisi Mallat db4 (a trous): W_j = x * Q_j, sudah disejajarkan (T_j + Delay_j = Tmax)')
    for j, w in enumerate(Wj, 1):
        axs[j].plot(n, w, lw=.8, color='tab:purple'); axs[j].set_ylabel(f'Q{j}', rotation=0, labelpad=14)
    axs[-1].set_xlabel('sampel (n)'); plt.tight_layout(); plt.show()

def plot_filterbank(res):
    fs = res['fs']
    fig, ax = plt.subplots(2, 3, figsize=(12, 6.5))
    ax[0, 0].stem(range(L), h_db4); ax[0, 0].set_title('h[n]  (LPF db4)')
    ax[0, 1].stem(range(L), g_db4); ax[0, 1].set_title('g[n] = (-1)^n h[L-1-n]  (HPF)')
    w = np.linspace(0, 1, len(_freq(h_db4)))
    ax[0, 2].plot(w, _freq(h_db4), label='|H|'); ax[0, 2].plot(w, _freq(g_db4), label='|G|')
    ax[0, 2].set(title='Respons frekuensi (sumbu = w/pi)'); ax[0, 2].legend()
    for j, q in enumerate(Q, 1):
        F = _freq(q, 8192); ax[1, 0].plot(np.linspace(0, fs / 2, len(F)), F, label=f'Q{j}')
    ax[1, 0].set(title='|Q_j(f)| ekuivalen filter bank', xlabel='Hz', xlim=(0, fs / 2)); ax[1, 0].legend(fontsize=7, ncol=2)
    for j in range(4):
        ax[1, 1].plot(Q[j], label=f'Q{j+1}')
    ax[1, 1].set(title='Impuls respons Q1..Q4 (j1..j4)'); ax[1, 1].legend(fontsize=7)
    ax[1, 2].axis('off')
    ax[1, 2].text(0, .95, 'Pemeriksaan koefisien db4\n\n'
        f'sum h   = {h_db4.sum():.6f}  (sqrt2 = {np.sqrt(2):.6f})\n'
        f'sum h^2 = {np.sum(h_db4**2):.6f}  (1)\n'
        f'sum g   = {g_db4.sum():.6f}  (0)\n\n'
        'T_j   = ' + str(T_auto) + '\nDelay = ' + str(D_auto), va='top', family='monospace', fontsize=9)
    plt.tight_layout(); plt.show()

def _mark(ax, n, w, idx, color, label):
    idx = np.array(idx, int)
    if len(idx):
        ax.plot(n[idx], w[idx], 'o', color=color, ms=5, label=label)

def plot_zerocross(res):
    n, Wj = _n(res), res['Wj']
    rows = [('Q3 : zero-crossing Q & S', Wj[2], [('Q', res['zc']['Q']), ('S', res['zc']['S'])], None),
            ('Q4 : zero-crossing R (+ ambang)', Wj[3], [('R', res['zc']['R'])], res['thr4']),
            ('Q5 : zero-crossing P (setelah QRS dipotong)', res['w5'], [('P', res['zc']['P'])], res['thr5']),
            ('Q6 : zero-crossing T (setelah QRS dipotong)', res['w6'], [('T', res['zc']['T'])], res['thr6'])]
    fig, axs = plt.subplots(len(rows), 1, figsize=(11, 9), sharex=True)
    for ax, (ttl, w, marks, thr) in zip(axs, rows):
        ax.plot(n, w, lw=.8, color='tab:purple'); ax.axhline(0, color='k', lw=.5)
        if thr:
            ax.axhline(thr, ls='--', c='r', lw=.8); ax.axhline(-thr, ls='--', c='r', lw=.8)
        for k, idx in marks:
            _mark(ax, n, w, idx, COL[k], f'zero-cross {k}')
        ax.set_title(ttl, fontsize=9); ax.legend(loc='upper right', fontsize=7); ax.grid(alpha=.3)
    axs[-1].set_xlabel('sampel (n)'); plt.tight_layout(); plt.show()

def plot_gradient(res):
    n, x, w4 = _n(res), res['x'], res['Wj'][3]
    fig, axs = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    axs[0].plot(n, x, lw=.8); axs[0].set_title('Sinyal ECG')
    axs[1].plot(n, np.gradient(x), lw=.8, color='tab:green'); axs[1].set_title('Gradien sinyal: dx/dn')
    axs[2].plot(n, w4, lw=.9, color='tab:purple', label='W4 (Q4)')
    axs[2].plot(n, np.gradient(w4) * 5, lw=.8, color='tab:orange', label='gradien W4 (x5)')
    for i, p in enumerate(res['pairsR']):
        axs[2].plot([n[p['e1']], n[p['e2']]], [w4[p['e1']], w4[p['e2']]], 'k^', ms=5, label='modulus maxima/minima' if i == 0 else None)
    axs[2].axhline(0, color='k', lw=.5); axs[2].set_title('Koefisien Q4, gradiennya, dan modulus maxima-minima pasangan R'); axs[2].legend(loc='upper right', fontsize=7)
    for a in axs: a.grid(alpha=.3)
    axs[-1].set_xlabel('sampel (n)'); plt.tight_layout(); plt.show()

def plot_detection(res):
    n, x = _n(res), res['x']
    fig, ax = plt.subplots(figsize=(12, 4.6))
    ax.plot(n, x, lw=.9, color='k')
    for key, c, lab in (('boxR', 'orange', 'pulsa QRS (Q4)'), ('boxP', 'tab:green', 'pulsa P (Q5)'), ('boxT', 'tab:cyan', 'pulsa T (Q6)')):
        for i, (a, b) in enumerate(res[key]):
            ax.axvspan(n[a], n[b], color=c, alpha=.22, label=lab if i == 0 else None)
    for k in 'PQRST':
        idx = np.array(res[k], int)
        if len(idx):
            ax.plot(n[idx], x[idx], 'o', color=COL[k], ms=6, label=k)
            for i in idx:
                ax.annotate(k, (n[i], x[i]), textcoords='offset points', xytext=(0, 7 if res['pol'] > 0 or k in 'PT' else -13),
                            ha='center', color=COL[k], fontsize=9, weight='bold')
    ttl = 'Deteksi P Q R S T - DWT db4 (from scratch)'
    if res['metric']:
        ttl += f"   |   R vs .atr: Se={res['metric'][0]*100:.1f}%  PPV={res['metric'][1]*100:.1f}%"
    ax.set(title=ttl, xlabel='sampel (n)', ylabel='mV'); ax.margins(y=0.15)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=8, fontsize=8); ax.grid(alpha=.3); plt.tight_layout()
    plt.show()
    print(f"R={len(res['R'])}  Q={len(res['Q'])}  S={len(res['S'])}  P={len(res['P'])}  T={len(res['T'])}")
    for k in 'PQRST':
        print(f'{k}:', [int(i) + res['off'] for i in res[k]])

TABS = [('Prosses', plot_input), ('Mallat', plot_mallat), ('Filter Bank', plot_filterbank),
        ('Zero-cross', plot_zerocross), ('Gradient', plot_gradient), ('Detection', plot_detection)]


In [30]:
# ============================================================
# 4. ANTARMUKA (ipywidgets) : panel samping + tab seperti aplikasi dosen
# ============================================================
import ipywidgets as wd
from IPython.display import display, clear_output

def _num(val, desc, kind=wd.FloatText, w='165px'):
    return kind(value=val, description=desc, layout=wd.Layout(width=w), style={'description_width': '62px'})

def build_ui(rec):
    w_rec = wd.Text(value=rec, description='Record', layout=wd.Layout(width='230px'), style={'description_width': '62px'})
    w_nd  = _num(999, 'NData', wd.IntText); w_st = _num(0, 'Start', wd.IntText); w_ch = _num(0, 'Channel', wd.IntText)
    w_T   = [_num(T_auto[j], f'T{j+1}', wd.IntText) for j in range(JMAX)]
    w_D   = [_num(D_auto[j], f'Delay{j+1}', wd.IntText) for j in range(JMAX)]
    w_auto = wd.Checkbox(value=True, description='T/Delay otomatis', indent=False)
    w_thR = _num(0.4, 'ThR', wd.FloatText); w_thP = _num(0.15, 'ThP', wd.FloatText); w_thT = _num(0.2, 'ThT', wd.FloatText)
    w_ref = _num(200, 'Refr ms', wd.IntText); w_pw = _num(300, 'Pwin ms', wd.IntText); w_tw = _num(450, 'Twin ms', wd.IntText)
    btn = wd.Button(description='Action', button_style='primary', layout=wd.Layout(width='230px'))
    status = wd.Output()
    outs = [wd.Output() for _ in TABS]
    tab = wd.Tab(children=outs)
    for i, (name, _) in enumerate(TABS):
        tab.set_title(i, name)

    def action(_=None):
        with status:
            clear_output(wait=True)
            try:
                nd, st = int(w_nd.value), int(w_st.value)
                x, fs, ref, name = load_record(w_rec.value, st, nd, int(w_ch.value))
                if w_auto.value:
                    for j in range(JMAX):
                        w_T[j].value, w_D[j].value = T_auto[j], D_auto[j]
                T = [int(v.value) for v in w_T]; D = [int(v.value) for v in w_D]
                P = dict(ThR=w_thR.value, ThP=w_thP.value, ThT=w_thT.value, Refr=w_ref.value,
                         Pwin=w_pw.value, Twin=w_tw.value, Start=st)
                res = analyze(x, fs, ref, P, T, D); res['name'] = name
                print('OK - fs =', fs, 'Hz, N =', len(x))
            except Exception as e:
                print('ERROR:', repr(e)); return
        for out, (_, fn) in zip(outs, TABS):
            with out:
                clear_output(wait=True)
                fn(res)

    btn.on_click(action)
    side = wd.VBox([wd.HTML('<b>Process</b>'), w_rec, w_nd, w_st, w_ch, btn, status,
                    wd.HTML('<b>T</b>')] + w_T + [wd.HTML('<b>Delay</b>')] + w_D +
                   [w_auto, wd.HTML('<b>Ambang & jendela</b>'), w_thR, w_thP, w_thT, w_ref, w_pw, w_tw],
                   layout=wd.Layout(width='250px'))
    display(wd.HBox([side, tab], layout=wd.Layout(width='100%')))
    action()

build_ui(REC)
